In [1]:
import os
import re
from pathlib import Path
from datetime import datetime

# Maths
import numpy as np
import pandas as pd
import seaborn as sns

# Database
from sqlalchemy import Column, Integer, BigInteger, Float, String, Boolean, DateTime, Date, ForeignKey
from sqlalchemy import create_engine, URL
from sqlalchemy.orm import declarative_base, relationship, Session, sessionmaker

# Helpful imports
from typing import Dict, Any, List
from IPython.display import clear_output, display

# Data scrapping
from bs4 import BeautifulSoup
import requests

# Data scrapping

In [ ]:
res = requests.get("https://opendata.gov.ua/dataset/air_monitor")
soup = BeautifulSoup(res.text, "html.parser")

regex = r"\d{4}-\d{2}-\d{2}"

for item in soup.find_all("div", {"class": "resource-item"}):
    name_block = item.find("div", {"class": "data-resource-name-content"})
    link_tag = name_block.find("a") if name_block else None

    if not link_tag:
        continue

    table_name_raw = link_tag.string or ""
    matches = re.findall(regex, table_name_raw)
    if not matches:
        continue

    table_name = f"{matches[0]}.csv"

    download_tag = item.find("a", {"class": "data-resource-download"}, href=True)
    if not download_tag:
        continue

    table_link = download_tag.get("href")

    file_downloaded = requests.get(table_link)
    print(file_downloaded.status_code, table_name, table_link)

    if file_downloaded.status_code == 200:
        with open(table_name, "wb") as file:
            file.write(file_downloaded.content)

# Передобробка даних

## Об'єднання даних

Дані було завантажено з сайту міськради використовуючи `scrapper.py` Оглянемо ці дані детальніше

Перш за все оглянемо чи всі таблиці мають однакові колонки. У випадку якщо ні — слід виконувати додаткові операції для коректного злиття даних

In [2]:
def check_if_columns_same(data_dir: str) -> bool:
    c_set = set()
    files = Path(data_dir).rglob('*.csv')
    for f in files:
        sdf = pd.read_csv(f)
        c_set.add(tuple(sdf.columns))

    return len(c_set) == 1


print(check_if_columns_same("./data"))

True


Як можна побачити — результат виконання функції `True`, тобто всі таблиці мають однакові колонки, що означає, таблиці можна з'єднувати без жодних проблем з сумісністю колонок. Для цього задамо спеціальну функцію

In [3]:
def combine_tables(data_dir: str) -> pd.DataFrame:
    files = Path(data_dir).rglob('*.csv')
    data_frame = pd.concat(map(pd.read_csv, files))
    return data_frame


df = combine_tables("./data")
df.head()

,stations_id,stations_name,Lat,Long,stations_time,stations_offset,stations_params_id,stations_params_key,stations_params_name,stations_params_localName,stations_params_unit,stations_params_localUnit,stations_params_value,stations_params_cr,stations_params_time,stations_params_offset,stations_params_level
0,767,Соборна 36,49.232859,28.470453,2022-10-25 13:24:28,0,33,SDS_P2,PM2.5,Пил 2.5 мкм,ug/m3,мкг/м³,9.14,1.000000,2022-10-25 13:24:28,0,1
1,767,Соборна 36,49.232859,28.470453,2022-10-25 13:24:28,0,34,SDS_P1,PM10,Пил 10 мкм,ug/m3,мкг/м³,11.69,1.000000,2022-10-25 13:24:28,0,1
2,767,Соборна 36,49.232859,28.470453,2022-10-25 13:24:28,0,7,CO2,CO₂,CO₂,ppm,мкг/м³,400.00,1.800009,2022-10-25 13:24:28,0,1
3,774,Станція замостя,49.245473,28.493727,2022-10-25 13:25:11,0,33,SDS_P2,PM2.5,Пил 2.5 мкм,ug/m3,мкг/м³,14.04,1.000000,2022-10-25 13:25:11,0,2
4,774,Станція замостя,49.245473,28.493727,2022-10-25 13:25:11,0,34,SDS_P1,PM10,Пил 10 мкм,ug/m3,мкг/м³,19.68,1.000000,2022-10-25 13:25:11,0,1


Як можна побачити набір даних успішно об'єднався, тому можна переходити до наступного етапу — вилучення надлишкових, дублікованих та суперечливих даних.

## Очищення даних

Оглянемо пропущені значення

In [4]:
df.isna().sum()

stations_id                        0
stations_name                      0
Lat                                0
Long                               0
stations_time                      0
stations_offset                    0
stations_params_id             41340
stations_params_key                0
stations_params_name               0
stations_params_localName          0
stations_params_unit               0
stations_params_localUnit          0
stations_params_value          86477
stations_params_cr           1765362
stations_params_time               0
stations_params_offset             0
stations_params_level              0
dtype: int64

Як можна побачити, велика кількість даних є пропущена в `stations_params_cr` та невелика частина в `stations_params_id` та `stations_params_value`. Оскільки якщо параметер не має ідентифікатора, або значення, він не несе корисної інформації його можна спокійно прибрати з набору дани

In [5]:
def prune_invalid_measurements(df: pd.DataFrame) -> pd.DataFrame:
    """
    Видаляє всі записи, де відсутнє значення виміру АБО оригінальний ID параметра.
    Забезпечує цілісність даних перед завантаженням у Fact_Measurements.
    """
    df = df.copy()
    initial_count = len(df)

    # Видаляє рядки, де хоча б одна з цих колонок має NaN
    df = df.dropna(subset=['stations_params_value', 'stations_params_id'])

    final_count = len(df)
    dropped_count = initial_count - final_count

    print(f"- Видалено рядків: {dropped_count}")
    print(f"- Залишилося для обробки: {final_count}")

    return df


df = prune_invalid_measurements(df)
df = df.replace(np.nan, None)

- Видалено рядків: 125924
- Залишилося для обробки: 7041122


Як можна побачити, незначна частина даних була видалена (приблизно 2%) але дані тепер позбавлені порожніх значень

Далі перевіримо набір даних на наявність станцій, що мають однакові ідентифікатори, проте, різні назви, щоб привести їх до першої нормальної форми

In [6]:
def identify_duplicate_names(df: pd.DataFrame) -> None:
    # Групуємо дані за ID та збираємо список усіх унікальних назв для кожного ID
    name_groups = df.groupby('stations_id')['stations_name'].unique()

    # Відфільтровуємо лише ті групи, де кількість унікальних назв більше 1
    problematic_stations = name_groups[name_groups.apply(len) > 1]

    # Перевірка на наявність розбіжностей
    if problematic_stations.empty:
        print("Перевірка назв завершена: розбіжностей не знайдено.")
        return

    # Вивід результатів
    print(f"Знайдено {len(problematic_stations)} ID з дубльованими назвами:\n")

    for station_id, names in problematic_stations.items():
        # Форматуємо масив назв у рядок для зручного читання
        names_list = ", ".join(map(str, names))
        print(f"ID: {station_id} | Назви: {names_list}")
        print("-" * 40)


identify_duplicate_names(df)

Знайдено 9 ID з дубльованими назвами:

ID: 90 | Назви: vinnytsia, vinnytsia-90
----------------------------------------
ID: 246 | Назви: vinnytsia, vinnytsia-246
----------------------------------------
ID: 256 | Назви: vinnytsia, vinnytsia-256
----------------------------------------
ID: 271 | Назви: vinnytsia, vinnytsia-271
----------------------------------------
ID: 274 | Назви: vinnytsia, vinnytsia-274
----------------------------------------
ID: 281 | Назви: vinnytsia, vinnytsia-281
----------------------------------------
ID: 315 | Назви: vinnytsia, vinnytsia-315
----------------------------------------
ID: 767 | Назви: Соборна 36, Хмельницьке шосе 27
----------------------------------------
ID: 1183 | Назви: Вишенька, Славне
----------------------------------------


Як можна побачити частина станцій мають конфліктуючі назви. Щоб це виправити потрібна додаткова функція

In [7]:
def remap_conflicting_names(data: pd.DataFrame) -> pd.DataFrame:
    stations_mapping = {
        256: "vinnytsia-256",
        281: "vinnytsia-281",
        315: "vinnytsia-315",
        90: "vinnytsia-90",
        271: "vinnytsia-271",
        767: "Соборна 36",
        1183: "Вишенька",
        246: "vinnytsia-246",
        274: "vinnytsia-274",
    }
    mask = df['stations_id'].isin(stations_mapping.keys())
    data.loc[mask, 'stations_name'] = data.loc[mask, 'stations_id'].map(stations_mapping)
    return data


df = remap_conflicting_names(df)
identify_duplicate_names(df)

Перевірка назв завершена: розбіжностей не знайдено.


Перевіримо, чи всі координати перетинаються та мають однакове походження

In [8]:
def check_coordinate_consistency(df: pd.DataFrame) -> None:
    # Групуємо за ID та рахуємо кількість унікальних значень широти й довготи
    stats = df.groupby('stations_id')[['Lat', 'Long']].nunique()

    # Відфільтровуємо лише ті ID, де кількість унікальних координат > 1
    problematic_mask = (stats['Lat'] > 1) | (stats['Long'] > 1)
    problematic_ids = stats[problematic_mask].index

    # Якщо проблемних станцій не знайдено — виходимо
    if problematic_ids.empty:
        print("Перевірка завершена: усі станції мають консистентні координати.")
        return

    # Виводимо деталі для кожної проблемної станції
    print(f"Знайдено {len(problematic_ids)} станцій з суперечливими координатами:\n")

    for sid in problematic_ids:
        # Отримуємо унікальні значення координат для цього конкретного ID
        unique_lats = df.loc[df["stations_id"] == sid, "Lat"].unique()
        unique_longs = df.loc[df["stations_id"] == sid, "Long"].unique()

        print(f"ID станції: {sid}")
        print(f"  - Унікальні Lat:  {unique_lats}")
        print(f"  - Унікальні Long: {unique_longs}")
        print("-" * 40)


check_coordinate_consistency(df)

Знайдено 10 станцій з суперечливими координатами:

ID станції: 92
  - Унікальні Lat:  [49.22665528295114 49.227702145800045]
  - Унікальні Long: [28.44795585 28.44476567]
----------------------------------------
ID станції: 256
  - Унікальні Lat:  [49.20485607380781 49.20875249539933]
  - Унікальні Long: [28.5288355  28.52855327]
----------------------------------------
ID станції: 767
  - Унікальні Lat:  [49.232859 '49.232859' 49.233663]
  - Унікальні Long: [28.470453 28.438175]
----------------------------------------
ID станції: 774
  - Унікальні Lat:  [49.245473 '49.245473']
  - Унікальні Long: [28.493727]
----------------------------------------
ID станції: 790
  - Унікальні Lat:  [49.227009 '49.227009' 49.2270093]
  - Унікальні Long: [28.418998  28.4189984]
----------------------------------------
ID станції: 1183
  - Унікальні Lat:  [49.227518 49.3331555]
  - Унікальні Long: [28.396157  28.5390885]
----------------------------------------
ID станції: 1315
  - Унікальні Lat:  [49

Оскільки існують станції з суперечними координатами, їх треба обробити. Для цього було розроблено спеціалізовану функцію, яка методом знаходження моди координат визначає найбільш імовірні координати станції

In [9]:
def standardize_coordinates(df: pd.DataFrame) -> None:
    """
    Очищує координати Lat/Long, конвертує їх у числові значення, округлює
    та замінює всі значення для кожної станції на їхню моду (найчастіше значення).
    """

    cols = ['Lat', 'Long']

    for col in cols:
        # Очищення та конвертація у числовий тип
        # Якщо в колонці є рядки, залишаємо лише цифри та крапку
        if df[col].dtype == 'object':
            df[col] = df[col].astype(str).str.extract(r'(\d+\.\d+)')[0]

        # Перетворюємо на float та округлюємо
        df[col] = pd.to_numeric(df[col], errors='coerce').round(5)

    # Визначення моди для кожного stations_id
    # Створюємо допоміжну функцію для отримання першої моди, щоб уникнути помилок,
    # якщо для групи немає моди (наприклад, усі значення NaN)
    def get_first_mode(series):
        m = series.mode()
        return m.iloc[0] if not m.empty else None

    df[cols] = df.groupby('stations_id')[cols].transform(get_first_mode)

    print("Координати успішно очищені та стандартизовані за модою.")


standardize_coordinates(df)
check_coordinate_consistency(df)

Координати успішно очищені та стандартизовані за модою.
Перевірка завершена: усі станції мають консистентні координати.


Далі розглянемо одиниці виміру

In [10]:
df["stations_params_key"].unique()

<ArrowStringArray>
[                'SDS_P2',                 'SDS_P1',                    'CO2',
     'BME280_temperature',        'BME280_humidity',        'BME280_pressure',
                    'pm0',                   'pm25',                   'pm10',
            'temperature',               'humidity',                 'PMS_P0',
                 'PMS_P2',                 'PMS_P1',     'ZPHS01B=pm1(ug/m3)',
    'ZPHS01B=pm25(ug/m3)',    'ZPHS01B=pm10(ug/m3)',   'AHTx0=temperature(C)',
  'BMP280=temperature(C)', 'ZPHS01B=temperature(C)',     'AHTx0=humidity(Rh)',
   'ZPHS01B=humidity(Rh)',    'BMP280=pressure(Pa)',       'ZPHS01B=co2(ppm)',
    'ZPHS01B=ch2o(mg/m3)',        'ZPHS01B=o3(ppm)',       'ZPHS01B=no2(ppm)',
        'ZPHS01B=co(ppm)',               'pressure',                   'PM10',
                  'PM2.5',             'VOC (H₂CO)',            'Temperature',
               'Humidity',               'Pressure',                  'PM1.0',
                     'CO',       

Як видно параметри станції, а саме те що вона вимірює абсолютно не стандартизован, мають різні форми та не годятться для подальшої роботи, тому вони потребують детального передобробленння. Для цього оголошено 3 допоміжні функції:
- `unify_measurement_units` – обробляє одиниці виміру
- `resolve_canonical_parameter` – обробляє канонічні назви вимірів
- `standardize_dimension_attributes` – з'єднувальна функція для двох попередніх

In [11]:
def get_canonical_parameters_map() -> Dict[str, str]:
    """
    Повний словник відповідності технічних кодів до канонічних назв.
    """
    return {
        # Пил
        'SDS_P1': 'PM10',
        'SDS_P2': 'PM2.5',
        'PMS_P0': 'PM1.0',
        'PMS_P1': 'PM10',
        'PMS_P2': 'PM2.5',
        'PM0': 'PM1.0',
        'PM1': 'PM1.0',
        'PM25': 'PM2.5',
        'PM100': 'PM10',
        'PM1.0': 'PM1.0',
        'PM2.5': 'PM2.5',
        'PM10': 'PM10',

        # Гази
        'CO2': 'CO2',
        'CO': 'CO',
        'NO2': 'NO2',
        'O3': 'O3',
        'O₃': 'O3',
        'NH3': 'NH3',
        'CH2O': 'HCHO',
        'H2CO': 'HCHO',
        'VOC': 'VOC',
        'NO₂': 'NO2',

        # Метео
        'TEMPERATURE': 'Temperature',
        'HUMIDITY': 'Humidity',
        'PRESSURE': 'Pressure',

        # Radiation (Радіація)
        'RAD': 'Radiation',

        # Specific Sensor
        'A4': 'CO',   # Часто CO на платах специфічних виробників
        'E1': 'NO2',  # Часто NO2
        'E3': 'O3',   # Часто O3
    }


In [12]:
def resolve_canonical_parameter(raw_key: Any) -> str:
    """
    Розпізнає канонічну назву параметра з технічного рядка будь-якої складності.
    """
    if pd.isna(raw_key) or str(raw_key).lower() == 'nan':
        return "UNKNOWN"

    # Очищення від назв вендорів та юнітів у дужках
    key = str(raw_key).strip()

    # Видалення всього до знаку '=' включно
    if '=' in key:
        key = key.split('=')[-1]

    # Видалення даних у дужках '(...)'
    key = re.sub(r'\(.*?\)', '', key)

    # Базове форматування даних
    key = key.replace(' ', '').replace('_', '').replace('.', '')
    key_upper = key.upper()
    param_map = get_canonical_parameters_map()

    # Заміна неправильних значень
    return param_map.get(key_upper, key_upper)


In [13]:
def unify_measurement_units(df: pd.DataFrame) -> pd.DataFrame:
    """
    Стандартизує одиниці виміру до єдиного Unicode-вигляду.
    """
    # Словник замін для одиниць
    unit_standardization = {
        'ug/m3': 'µg/m³',
        'мкг/м³': 'µg/m³',
        'ug/m³': 'µg/m³',
        'ppm': 'ppm',
        'ppb': 'ppb',
        '%': '%',
        'Rh': '%',
        '°C': '°C',
        'C': '°C',
        'Pa': 'Pa',
        'hPa': 'hPa',
        'mg/m3': 'mg/m³',
        'uSv/h': 'µSv/h'
    }

    def clean_unit(unit_str):
        if pd.isna(unit_str): return 'unknown'
        u = str(unit_str).strip()
        # Видаляє дужки, якщо вони прийшли з даними
        u = u.replace('(', '').replace(')', '')
        return unit_standardization.get(u, u)

    df['stations_params_unit'] = df['stations_params_unit'].apply(clean_unit)
    return df

In [14]:
def standardize_dimension_attributes(df: pd.DataFrame) -> pd.DataFrame:
    """
    Оркестратор попередніх функцій
    """
    df = df.copy()
    df['stations_params_key'] = df['stations_params_key'].apply(resolve_canonical_parameter)
    df = unify_measurement_units(df)
    df['stations_params_name'] = df['stations_params_name'].fillna(df['stations_params_key'])
    return df


df = standardize_dimension_attributes(df)

## Передоброблення часу

Набір даних має виняткові ситуації, де час записано неправильно, щоб уникнути подальших поилок його слід обробити до спільного формату.

In [15]:
def prepare_dataframe_dates(df: pd.DataFrame) -> pd.DataFrame:
    """
    Попередня обробка датафрейму, щоб переконатися, що всі стовпці дати
    мали спільний формат. Підтримує ISO8601 та змішані формати.
    """
    date_cols = ['stations_time', 'stations_params_time']
    for col in date_cols:
        # format='mixed' для винятків 
        # '2022-10-25 13:24:28' and '2023-07-03T12:10:00Z'
        df[col] = pd.to_datetime(df[col], format='mixed', utc=True)
    return df


df = prepare_dataframe_dates(df)

# Створення моделі бази даних

Глянемо на дані які було отримано в результаті обробки, а точніше на поля та їх типи данимз

In [16]:
df.dtypes

stations_id                                int64
stations_name                                str
Lat                                      float64
Long                                     float64
stations_time                datetime64[us, UTC]
stations_offset                            int64
stations_params_id                        object
stations_params_key                          str
stations_params_name                         str
stations_params_localName                    str
stations_params_unit                         str
stations_params_localUnit                    str
stations_params_value                    float64
stations_params_cr                        object
stations_params_time         datetime64[us, UTC]
stations_params_offset                     int64
stations_params_level                      int64
dtype: object

Як видно з оброблених даних — більшість об'єктів залишаються типос object, тобто текстові дані. Для того щоб краще працювати з ними слід створити базу даних, використовуючи стандартну схему Data Warehouse, а саме – модель "Сніжинка".

В даному випадку будуть настунпі таблиці:
- **Фактологічна таблиця**
    - Виміри — поєднує решту таблиць бази даних
- **Вимірні таблиці**
    - Параметри – зберігає змінні параметри вимірів, такі як відхилення
    - Станції — зберігає дані про станцію та її властивості
    - Одиниці виміру — зберігає дані про одиниці виміру

![](./uml/schema.png)

## Реалізаціія схеми даних використовуючи SQLalchemy

In [17]:
Base = declarative_base()  # створення бази для задання наступних моделей

### Заданя вимірної таблиці вимірів

In [18]:
class DimUnit(Base):
    """
    Таблиця одиниць виміру, вимірна таблиця
    """
    __tablename__ = 'dim_units'

    unit_key = Column(Integer, primary_key=True, autoincrement=True)
    unit_name = Column(String(64), nullable=False)
    unit_symbol = Column(String(8))
    local_unit_name = Column(String(64))

    # Зв'язки
    parameters = relationship("DimParameter", back_populates="unit")

### Задання таблиць параметрів

In [19]:
class DimParameter(Base):
    """
    Таблиця параметрів вимірів, вимірна таблиця
    """
    __tablename__ = 'dim_parameters'

    parameter_key = Column(Integer, primary_key=True, autoincrement=True)
    parameter_code = Column(String(64), nullable=False)
    parameter_name = Column(String(64))
    local_name = Column(String(64))
    unit_key = Column(Integer, ForeignKey('dim_units.unit_key'))

    # SCD Type 2 колонки
    valid_from = Column(DateTime, nullable=False)
    valid_to = Column(DateTime)
    is_current = Column(Boolean, default=True)

    # Зв'язки
    unit = relationship("DimUnit", back_populates="parameters")
    measurements = relationship("FactMeasurement", back_populates="parameter")

### Задання таблиць танцій

In [20]:
class DimStation(Base):
    """
    Таблиця станцій, вимірна таблиця
    """
    __tablename__ = 'dim_stations'

    station_key = Column(Integer, primary_key=True, autoincrement=True)
    station_id = Column(Integer)
    station_name = Column(String(128))
    latitude = Column(Float)
    longitude = Column(Float)
    timezone_offset = Column(Integer)

    # SCD Type 2 колонки
    valid_from = Column(DateTime, nullable=False)
    valid_to = Column(DateTime)
    is_current = Column(Boolean, default=True)

    # Зв'язки
    measurements = relationship("FactMeasurement", back_populates="station")

### Задання фактологічної таблиці

In [21]:
class FactMeasurement(Base):
    """
    Фактологічна таблиця
    """
    __tablename__ = 'fact_measurements'

    measurement_id = Column(BigInteger, primary_key=True, autoincrement=True)

    # Зовнішні ключи
    station_key = Column(Integer, ForeignKey('dim_stations.station_key'), nullable=False)
    parameter_key = Column(Integer, ForeignKey('dim_parameters.parameter_key'), nullable=False)

    # Виміри
    value = Column(Float)
    quality_ratio = Column(Float, nullable=True)
    pollution_level = Column(Integer)

    # Метаданні (Час інтегровано сюди)
    measurement_timestamp = Column(DateTime, nullable=False, index=True)
    offset_minutes = Column(Integer)

    # Зв'язки
    station = relationship("DimStation", back_populates="measurements")
    parameter = relationship("DimParameter", back_populates="measurements")

## Створення фізичної моделі

Примітка стосовно `host` — його треба замінити на 
- `127.0.0.1` для підключення до локальної mysql бази даних
- назву Docer Compose сервісу якщо запускається там
- посилання на віддалений сервер за потреби

In [22]:
engine = create_engine(
    URL.create(
        drivername="mysql",
        username=os.getenv("MYSQL_USER"),
        password=os.getenv("MYSQL_PASSWORD"),
        host="db",
        port=3306,
        database=os.getenv("MYSQL_DATABASE")
    )
)

In [23]:
Base.metadata.create_all(engine)

# Заповнення бази данних даними

Розглянемо голову набору даних ще раз

In [24]:
df.head()

,stations_id,stations_name,Lat,Long,stations_time,stations_offset,stations_params_id,stations_params_key,stations_params_name,stations_params_localName,stations_params_unit,stations_params_localUnit,stations_params_value,stations_params_cr,stations_params_time,stations_params_offset,stations_params_level
0,767,Соборна 36,49.23286,28.47045,2022-10-25 13:24:28+00:00,0,33,SDSP2,PM2.5,Пил 2.5 мкм,µg/m³,мкг/м³,9.14,1.0,2022-10-25 13:24:28+00:00,0,1
1,767,Соборна 36,49.23286,28.47045,2022-10-25 13:24:28+00:00,0,34,SDSP1,PM10,Пил 10 мкм,µg/m³,мкг/м³,11.69,1.0,2022-10-25 13:24:28+00:00,0,1
2,767,Соборна 36,49.23286,28.47045,2022-10-25 13:24:28+00:00,0,7,CO2,CO₂,CO₂,ppm,мкг/м³,400.00,1.800009,2022-10-25 13:24:28+00:00,0,1
3,774,Станція замостя,49.24547,28.49373,2022-10-25 13:25:11+00:00,0,33,SDSP2,PM2.5,Пил 2.5 мкм,µg/m³,мкг/м³,14.04,1.0,2022-10-25 13:25:11+00:00,0,2
4,774,Станція замостя,49.24547,28.49373,2022-10-25 13:25:11+00:00,0,34,SDSP1,PM10,Пил 10 мкм,µg/m³,мкг/м³,19.68,1.0,2022-10-25 13:25:11+00:00,0,1


Як можна побачити — дані не можна просто перенести до бази даних не вносячи жодних змін. Замість цього весь процес слід розбити на кроки та об'єднати їх у ETL pipline.

Кроки пайплайну повинні бути наступними:
- Трансформувати та завантажити одиниці виміру
- Трансформувати та завнтажити параметри вимірів
- Трансформувати та завантажити дані про станції
- Трансформувати та завнтажити дані про часи вимірів
- Сформувати таблицю фактів

Надалі наведено кроки пайплайну

## Трансформація та завантаження одиниць виміру

In [25]:
def transform_and_load_units(df: pd.DataFrame, session: Session) -> Dict[str, int]:
    """
    Витягує унікальні одиниці з фрейму даних та завантажує їх у DimUnits.
    Повертає відображення {unit_symbol: unit_key}.
    """
    # Ідентифікувати унікальні одиниці виміру
    unique_units = df[['stations_params_unit', 'stations_params_localUnit']].drop_duplicates()

    unit_map = {}
    for _, row in unique_units.iterrows():
        # Пabsеревірити наявність одиниць виміру в таблиці
        # необхідно для уникнення дублювання даних
        unit = session.query(DimUnit).filter_by(unit_symbol=row['stations_params_unit']).first()
        if not unit:
            unit = DimUnit(
                unit_name=row['stations_params_unit'], # використати символ як назву
                unit_symbol=row['stations_params_unit'],
                local_unit_name=row['stations_params_localUnit']
            )
            session.add(unit)
            session.flush()
        unit_map[unit.unit_symbol] = unit.unit_key

    return unit_map


## Трансформація та завантаження параметрів виміру

In [26]:
def transform_and_load_parameters(
    df: pd.DataFrame,
    session: Session,
    unit_map: Dict[str, int]
) -> Dict[str, int]:
    """
    Витягує унікальні параметри та завантажує їх у DimParameters.
    Обробляє посилання нормалізації на DimUnits.
    """
    # Аналогічно до поперднього отримує унікальні значення
    unique_params = df[[
        'stations_params_key', 'stations_params_name',
        'stations_params_localName', 'stations_params_unit'
    ]].drop_duplicates()

    param_map = {}
    for _, row in unique_params.iterrows():
        param = session.query(DimParameter).filter_by(parameter_code=row['stations_params_key']).first()
        if not param:
            param = DimParameter(
                parameter_code=row['stations_params_key'],
                parameter_name=row['stations_params_name'],
                local_name=row['stations_params_localName'],
                unit_key=unit_map.get(row['stations_params_unit']),
                valid_from=datetime.now(),
                is_current=True
            )
            session.add(param)
            session.flush()
        param_map[param.parameter_code] = param.parameter_key

    return param_map


## Трансформація та завантаження даних про станції

In [27]:
def transform_and_load_stations(df: pd.DataFrame, session: Session) -> Dict[int, int]:
    """
    Витягує унікальні станції та завантажує їх у DimStations.
    Повертає зіставлення {business_station_id: surrogate_station_key}.
    """
    unique_stations = df[['stations_id', 'stations_name', 'Lat', 'Long', 'stations_offset']].drop_duplicates()

    station_map = {}
    for _, row in unique_stations.iterrows():
        station = session.query(DimStation).filter_by(station_id=row['stations_id'], is_current=True).first()
        if not station:
            station = DimStation(
                station_id=row['stations_id'],
                station_name=row['stations_name'],
                latitude=row['Lat'],
                longitude=row['Long'],
                timezone_offset=row['stations_offset'],
                valid_from=datetime.now(),
                is_current=True
            )
            session.add(station)
            session.flush()
        station_map[station.station_id] = station.station_key

    return station_map

## Формування таблиці фактів

In [28]:
def load_fact_measurements(
    df: pd.DataFrame,
    session: Session,
    station_map: Dict[int, int],
    param_map: Dict[str, int],
    batch_size: int = 5000,
) -> None:
    """
    Ітерує по датафрейму та заповнює таблицю Fact_Measurements.
    """
    batch_buffer: List[FactMeasurement] = []

    n_added = 0
    df_size = len(df['stations_params_value'])
    for index, row in df.iterrows():
        ts = row['stations_params_time']
        if pd.isna(ts):
            continue

        # Отримання сурогатних ключів з мап
        s_key = station_map.get(row['stations_id'])
        p_key = param_map.get(row['stations_params_key'])

        # Пропуск запису, якщо ключів немає в словниках
        if s_key is None or p_key is None:
            continue

        # Створення екземпляру моделі
        fact = FactMeasurement(
            station_key=s_key,
            parameter_key=p_key,
            value=row['stations_params_value'],
            quality_ratio=row['stations_params_cr'],
            pollution_level=row['stations_params_level'],
            measurement_timestamp=ts,
            offset_minutes=row['stations_params_offset']
        )
        batch_buffer.append(fact)

        # Коли буфер досягає ліміту, виконуємо пакетне завантаження
        if len(batch_buffer) >= batch_size:
            session.bulk_save_objects(batch_buffer)
            session.commit()

            # Очищення буфера та звільнення пам'яті від об'єктів у сесії
            batch_buffer.clear()
            session.expunge_all()
            n_added += batch_size
            print(f"\rДодано: {n_added}/{df_size}", end="")

    # Завантаження залишків даних, що не увійшли в останній повний батч
    if batch_buffer:
        session.bulk_save_objects(batch_buffer)
        session.commit()
        session.expunge_all()
        batch_buffer.clear()


## Запуск пайплайну

Задання пайплайну

In [29]:
def run_pipeline(df: pd.DataFrame, session: Session) -> None:
    """
    Оркеструє процес трансформації та завантаження.
    """
    try:
        print("Обробка одиниць виміру...")
        u_map = transform_and_load_units(df, session)

        print("Обробка параметрів виміру...")
        p_map = transform_and_load_parameters(df, session, u_map)

        print("Обробка станцій...")
        s_map = transform_and_load_stations(df, session)

        print("Завантаження таблиці фактів...")
        load_fact_measurements(df, session, s_map, p_map)

        print("\nETL пайплайн завантажено.")
    except Exception as e:
        session.rollback()
        print(f"Помилка під час ETL: {e}")
        raise e

Створення сесії взаємодії з базою даних

In [30]:
Session = sessionmaker(engine)
session = Session()

Запуск пайплайну

In [31]:
run_pipeline(df, session)

Обробка одиниць виміру...
Обробка параметрів виміру...
Обробка станцій...
Завантаження таблиці фактів...
Додано: 7040000/7041122
ETL пайплайн завантажено.
